# 10 | RAG最小问答链路：Prompt组装与LLM调用

上一篇讲到 Retriever：

```text
用户问题 -> Retriever -> 相关 Documents
```

但 Documents 不能直接变成回答。它们还需要被整理成 Prompt 里的上下文，再交给模型。

这篇就讲 RAG 里很关键的一步：

```text
用户问题 + 检索结果 -> Prompt
```

说得再具体一点：

```text
question + retrieved_docs -> context -> ChatPromptTemplate
```

这一步看起来不炫，但很重要。资料塞得乱，模型就答得乱。就像你让同事写报告，却把资料截图、表格、聊天记录全扔给他，不崩溃算他职业素养好。


# 一、Prompt 组装在 RAG 里的位置

现在 RAG 链路走到这里了：

```text
资料 -> Loader -> Text Splitter -> Vector Store -> Retriever -> Documents -> Prompt -> LLM
```

Retriever 只负责把相关 `Document` 找出来。

Prompt 组装负责把这些 `Document` 变成模型能读懂的上下文。

这一步通常要做三件事：

1. 提取每个 Document 的正文。
2. 拼成一段清晰的 context。
3. 和用户问题一起放进 Prompt。


# 二、准备一个 Retriever

继续用客服知识库场景。

用户问：

```text
未发货订单能退款吗？
```

Retriever 负责查资料，Prompt 负责把查到的资料和问题一起组织好。


In [ ]:
from pathlib import Path

# 示例文件放在 rag/data 目录。
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

kb_path = data_dir / "support_policy_prompt.txt"

# 准备一份客服知识库文本。
# 这一份资料后面会被加载、切分、检索，并最终进入 Prompt。
kb_path.write_text(
    "售后知识库说明\n\n"
    "一、未发货订单退款\n"
    "如果订单还没有发货，用户可以在订单详情页申请退款。系统会自动拦截发货流程，退款通常会在 1 到 3 个工作日内原路退回。"
    "如果订单已经进入仓库打包状态，退款申请可能需要人工审核。\n\n"
    "二、已发货订单退款\n"
    "如果订单已经发货，用户需要先等待商品送达，再根据实际情况申请退货退款。"
    "退货时需要保持商品包装完整，并上传物流单号。仓库签收并验货通过后，退款会在 3 到 7 个工作日内退回。\n\n"
    "三、发票下载\n"
    "电子发票可以在订单详情页下载。如果用户找不到发票入口，可以引导用户进入：我的订单 -> 订单详情 -> 发票信息。"
    "如果发票抬头填写错误，需要联系人工客服重新开具。\n\n"
    "四、会员自动续费\n"
    "会员服务默认不会强制续费。用户如果开通了自动续费，可以在 App 的会员中心关闭。"
    "关闭自动续费后，已经购买的会员权益不会立即失效，会持续到当前会员周期结束。\n",
    encoding="utf-8",
)

print(kb_path)


In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings

# 1. 把本地文本加载成 Document。
loader = TextLoader(str(kb_path), encoding="utf-8")
documents = loader.load()

# 2. 把长 Document 切成更适合检索的小 chunks。
splitter = RecursiveCharacterTextSplitter(
    chunk_size=120,
    chunk_overlap=20,
    separators=["\n\n", "。", "，", " ", ""],
    add_start_index=True,
)
chunks = splitter.split_documents(documents)

# 3. 创建 Embedding 模型，用于把 chunks 转成向量。
embeddings = OllamaEmbeddings(
    model="qwen3-embedding:latest",
    base_url="http://localhost:11434",
)

# 4. 把 chunks 写入向量库。
vectorstore = InMemoryVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
)

# 5. 把向量库转换成 Retriever。
# 后面组 Prompt 时，不直接操作 vectorstore，而是通过 retriever 拿资料。
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})


到这里，准备工作就完成了。

注意这篇的重点不是重新讲 loader、splitter、vectorstore，而是从 Retriever 返回结果开始，往 Prompt 里装。


# 三、先拿到检索结果

Retriever 返回的是 `Document` 列表。

先看它实际拿回了什么。


In [ ]:
# 用户的原始问题。
question = "未发货订单能退款吗？"

# Retriever 只负责查资料：输入问题，返回相关 Document 列表。
retrieved_docs = retriever.invoke(question)

print("返回 Document 数量:", len(retrieved_docs))
for i, doc in enumerate(retrieved_docs, start=1):
    print(f"# Document {i}")
    print(doc.page_content)
    print(doc.metadata)
    print("-" * 40)


这些内容还不能直接丢给模型。

我们需要先把多个 Document 整理成一个 `context` 字符串。

否则 Prompt 里就会像快递箱没拆封：东西都在，但不好用。


# 四、把 Documents 格式化成 context

最常见的做法是：取出每个 `Document.page_content`，用分隔符拼起来。

如果需要展示来源，也可以把 metadata 一起拼进去。


In [ ]:
def format_docs(docs):
    """把 Retriever 返回的 Document 列表整理成 Prompt 里的 context。"""
    formatted = []

    for i, doc in enumerate(docs, start=1):
        # metadata 不是必须给模型看，但保留下来有利于追踪来源。
        source = doc.metadata.get("source", "unknown")
        start_index = doc.metadata.get("start_index", "unknown")

        # 每个 Document 单独编号，避免多段资料混在一起。
        formatted.append(
            f"[资料{i}]\n"
            f"来源: {source}\n"
            f"位置: {start_index}\n"
            f"内容: {doc.page_content}"
        )

    # 用空行分隔不同资料，让 Prompt 更清晰。
    return "\n\n".join(formatted)

# context 是真正会塞进 Prompt 的资料区。
context = format_docs(retrieved_docs)

print(context)


`context` 就是 RAG Prompt 里的“资料区”。

这里建议保留资料编号，比如 `[资料1]`、`[资料2]`。

后面让模型回答时，可以要求它基于这些资料回答。这样模型不容易自由发挥，至少不会一开口就像在写玄幻小说。


# 五、创建 ChatPromptTemplate

现在用 `ChatPromptTemplate` 组装 Prompt。

当前 LangChain 推荐使用 `langchain_core.prompts.ChatPromptTemplate`。

这里不使用旧的 `RetrievalQA`，而是把每一步拆开：检索、格式化、组 Prompt。这样更清楚，也更容易调试。


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        # system 消息用来规定模型的回答边界。
        # 这里明确要求：只能基于资料回答，资料没有就说没有。
        "你是一个严谨的客服知识库助手。"
        "只能基于提供的资料回答问题。"
        "如果资料里没有答案，就说：资料中没有找到明确说明。"
    ),
    (
        "human",
        # human 消息里放两个变量：用户问题 question，以及检索资料 context。
        "用户问题：{question}\n\n"
        "可用资料：\n{context}\n\n"
        "请用简洁中文回答。"
    ),
])

prompt


这个 Prompt 有两个变量：

- `question`：用户问题
- `context`：Retriever 查回来的资料

这就是 RAG Prompt 的核心结构。


# 六、把 question 和 context 填进 Prompt

先不调用模型，只看最终消息长什么样。

这一步很适合调试。Prompt 组装错了，后面模型再强也只能接锅。


In [ ]:
# invoke 会把 question/context 填进 Prompt 模板。
# 这里还没有调用 LLM，只是在生成最终要发给模型的消息。
messages = prompt.invoke({
    "question": question,
    "context": context,
})

# 直接打印 messages 对象不太直观，所以按消息类型展开看。
for i, message in enumerate(messages.messages, start=1):
    print(f"===== Message {i}: {message.type.upper()} =====")
    print(message.content)
    print()


到这里，我们已经完成了：

```text
用户问题 + 检索结果 -> Prompt Messages
```

这一步之后，才轮到 LLM 出场。

Prompt 就像给模型准备的资料夹：问题放上面，证据放下面，别让它去脑海里临场编故事。


# 七、用 LCEL 串起来

手动写一遍很适合理解流程。

实际工程里，可以用 LCEL 把这些步骤串成一个小链：

```text
question -> retriever -> format_docs -> prompt
```

先直接看代码。它一开始会有点绕，后面再用图拆开。


In [ ]:
from langchain_core.runnables import RunnablePassthrough

prompt_chain = {
    # context 分支：问题先进入 retriever，拿到 Documents，再用 format_docs 整理成字符串。
    "context": retriever | format_docs,
    # question 分支：用户问题原样传入 Prompt。
    "question": RunnablePassthrough(),
} | prompt

# 输入一个问题，链路会自动完成：检索 -> 格式化 -> 组装 Prompt。
prompt_messages = prompt_chain.invoke("未发货订单能退款吗？")

for i, message in enumerate(prompt_messages.messages, start=1):
    print(f"===== Message {i}: {message.type.upper()} =====")
    print(message.content)
    print()


如果刚才这段 LCEL 看起来有点绕，可以按下面这张图理解。

同一个用户问题会分成两条线走：

```mermaid
graph TD
    A["用户问题
未发货订单能退款吗？"] --> B["context 分支"]
    A --> C["question 分支"]

    B --> D["Retriever
检索 Documents"]
    D --> E["format_docs
整理成 context"]

    C --> F["RunnablePassthrough
保留原问题"]

    E --> G["ChatPromptTemplate"]
    F --> G

    G --> H["Prompt Messages
SYSTEM + HUMAN"]
```

对应代码就是：

```python
{
    "context": retriever | format_docs,
    "question": RunnablePassthrough(),
} | prompt
```

可以把它理解成：

- `context` 这一路：拿问题去查资料，然后整理成上下文
- `question` 这一路：问题本身不动，原样放进 Prompt
- 最后两路在 `prompt` 里汇合

所以这里不是两个问题，也不是执行两次问答。只是同一个输入，被拆成两份用途：一份用来查资料，一份用来问模型。


这段链路读起来就是：

```text
输入问题
  -> context 分支：检索资料，再格式化成 context
  -> question 分支：原样保留用户问题
  -> 两路一起进入 Prompt
```

这就是 RAG 里非常常见的写法。

真正调用模型时，只需要在后面继续接 LLM 和输出解析器。


# 八、如果要接 LLM，链路长这样

这一步先给完整形态，不强制运行。

因为不同同学本地模型、API Key、模型服务可能不一样。


In [ ]:
# 示例：如果你本地有可用的聊天模型，可以继续这样接。
# 这里先注释掉，避免把本文重点从“组装 Prompt”提前拉到“完整问答链”。

from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model("ollama:qwen2.5:14b")

rag_chain = prompt_chain | llm | StrOutputParser()
answer = rag_chain.invoke("未发货订单能退款吗？")
print(answer)


这里刻意不使用 `RetrievalQA` 这种旧式封装。

现在更推荐把 RAG 拆成清楚的几个 Runnable：

```text
Retriever -> format_docs -> Prompt -> LLM -> Parser
```

拆开以后，哪一步出问题都能看得见。检索错了查 Retriever，资料拼错了查 format_docs，回答跑偏了再看 Prompt。别一上来就怪模型，它也挺委屈。


## 小结

这篇只记住几件事：

1. Retriever 返回的是 `Document` 列表。
2. Prompt 不能直接吃一堆杂乱 Document，最好先格式化成 `context`。
3. `format_docs()` 常用来把 Documents 拼成上下文。
4. `ChatPromptTemplate` 负责把 `question` 和 `context` 变成消息。
5. 当前推荐用 Runnable / LCEL 把链路串起来。
6. 不建议用旧式 `RetrievalQA` 当主线。

RAG 链路现在走到这里：

```text
资料 -> Loader -> Splitter -> Vector Store -> Retriever -> Prompt
```

下一步，就是让 LLM 基于这个 Prompt 生成最终回答。
